# Periodic committor experiment

This notebook is self-contained. You do **not** need to contact Codex for ordinary runs.

1. Run the first code cell once in each fresh Colab runtime. It downloads the private repository and runs CPU-only tests; it does not train a model.
2. In the second code cell, set `RUN_MODE` to either `"check"`, `"pilot"`, or `"resume"` and run that cell.
3. Run the final cell only after a `pilot` or `resume` run finishes.

`check` is the safe default and uses no GPU time. `pilot` is the actual three-method experiment: it trains three PINNs for 6,000 epochs each and therefore requires a GPU. `resume` continues an interrupted pilot from its Drive output folder.

Before using `pilot` or `resume`, choose **Runtime → Change runtime type → T4 GPU → Save**. The token requested below is used only to clone this private repository; it is not saved in the repository.

In [ ]:
from pathlib import Path
from getpass import getpass
import base64
import shutil
import subprocess

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = 'committor-colab'

if not (REPO / 'pyproject.toml').is_file():
    if REPO.exists():
        # The fixed path contains only an incomplete clone from this notebook.
        shutil.rmtree(REPO)
    token = getpass('Paste your GitHub token, then press Enter: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    try:
        subprocess.run(
            ['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}',
             'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
    finally:
        del token, auth
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
else:
    print('Using the existing clone at', REPO)

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
assert shutil.which('met-pinn-periodic-comparison'), 'The experiment command was not installed.'
print('Setup passed. GPU available:', __import__('torch').cuda.is_available())

In [ ]:
# Choose exactly one mode, then run this cell.
RUN_MODE = 'check'      # 'check', 'pilot', or 'resume'
RESUME_RUN_DIR = None   # Required only for 'resume'; paste a prior Drive folder here.

from datetime import datetime
from pathlib import Path
import subprocess

CONFIG = 'configs/periodic_committor/overnight_comparison.json'

if RUN_MODE == 'check':
    RUN_ROOT = None
    print('Check complete. No model was trained and no GPU time was used.')
elif RUN_MODE == 'pilot':
    if not __import__('torch').cuda.is_available():
        raise RuntimeError('Pilot mode requires a GPU. Select a T4 GPU under Runtime → Change runtime type.')
    run_name = 'pilot_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
    Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
    print('Starting the full three-method pilot. Results will be saved in:', RUN_ROOT)
    subprocess.run(['met-pinn-periodic-comparison', CONFIG, '--output-dir', RUN_ROOT], check=True)
elif RUN_MODE == 'resume':
    if not __import__('torch').cuda.is_available():
        raise RuntimeError('Resume mode requires a GPU. Select a T4 GPU under Runtime → Change runtime type.')
    if not RESUME_RUN_DIR:
        raise ValueError('For resume mode, set RESUME_RUN_DIR to the Drive folder printed by the earlier pilot.')
    RUN_ROOT = RESUME_RUN_DIR
    print('Resuming:', RUN_ROOT)
    subprocess.run(['met-pinn-periodic-comparison', CONFIG, '--output-dir', RUN_ROOT, '--resume'], check=True)
else:
    raise ValueError("RUN_MODE must be 'check', 'pilot', or 'resume'.")

In [ ]:
import json
from IPython.display import Image, display

if RUN_ROOT is None:
    print('There are no figures in check mode. Set RUN_MODE to \'pilot\' when you want to train the full comparison.')
else:
    with open(f'{RUN_ROOT}/summary.json') as handle:
        print(json.dumps(json.load(handle), indent=2))
    for filename in ['losses.png', 'committor_boundary_adapted.png', 'committor_naive_product.png', 'committor_penalty.png']:
        display(Image(filename=f'{RUN_ROOT}/{filename}'))